In [2]:
# PHASE 9 — Supply / Replenishment Planning
# Step 9.0: Load the cleaned dataset

import pandas as pd
import numpy as np

file_path = "cleaned_supply_chain_dataset.csv"

df = pd.read_csv(file_path)

df["date"] = pd.to_datetime(
    df["date"],
    format="%d-%m-%Y",
    errors="raise"
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())

Dataset loaded successfully!
Dataset shape: (91250, 34)
Date range: 2024-01-01 00:00:00 to 2024-12-30 00:00:00


In [3]:
# Step 9.1: Get the latest inventory position for each SKU-Warehouse

latest_inventory = (
    df
    .sort_values("date")
    .groupby(["sku_id", "warehouse_id"])
    .tail(1)
    [
        [
            "sku_id",
            "warehouse_id",
            "date",
            "inventory_level"
        ]
    ]
    .rename(
        columns={
            "date": "inventory_as_of_date",
            "inventory_level": "current_inventory"
        }
    )
    .reset_index(drop=True)
)

print("Latest Inventory Position\n")

print("SKU-Warehouse combinations:", len(latest_inventory))

print(
    "Latest inventory date:",
    latest_inventory["inventory_as_of_date"].min(),
    "to",
    latest_inventory["inventory_as_of_date"].max()
)

display(latest_inventory.head(10))

Latest Inventory Position

SKU-Warehouse combinations: 250
Latest inventory date: 2024-12-30 00:00:00 to 2024-12-30 00:00:00


,sku_id,warehouse_id,inventory_as_of_date,current_inventory
0,SKU_25,WH_5,2024-12-30,373
1,SKU_50,WH_3,2024-12-30,369
2,SKU_18,WH_5,2024-12-30,766
3,SKU_18,WH_4,2024-12-30,339
4,SKU_18,WH_3,2024-12-30,442
5,SKU_18,WH_2,2024-12-30,272
6,SKU_18,WH_1,2024-12-30,407
7,SKU_17,WH_5,2024-12-30,334
8,SKU_17,WH_4,2024-12-30,362
9,SKU_17,WH_3,2024-12-30,574


In [4]:
# Step 9.2A: Load forecast and calculate 30-day demand by SKU-Warehouse

forecast_output = pd.read_excel(
    "30_Day_SKU_Warehouse_Forecast.xlsx",
    sheet_name="30_Day_Forecast"
)

forecast_summary = (
    forecast_output
    .groupby(["sku_id", "warehouse_id"])
    .agg(
        forecast_daily_demand=("forecast_demand", "mean"),
        total_forecast_demand=("forecast_demand", "sum")
    )
    .reset_index()
)

print("Forecast Summary")
print("Forecast rows:", len(forecast_output))
print("SKU-Warehouse combinations:", len(forecast_summary))

display(forecast_summary.head(10))

Forecast Summary
Forecast rows: 7500
SKU-Warehouse combinations: 250


,sku_id,warehouse_id,forecast_daily_demand,total_forecast_demand
0,SKU_1,WH_1,17.833333,535.00
1,SKU_1,WH_2,19.477667,584.33
2,SKU_1,WH_3,21.433000,642.99
3,SKU_1,WH_4,22.148667,664.46
4,SKU_1,WH_5,23.299667,698.99
5,SKU_10,WH_1,19.185333,575.56
6,SKU_10,WH_2,21.119333,633.58
7,SKU_10,WH_3,21.469000,644.07
8,SKU_10,WH_4,21.599333,647.98
9,SKU_10,WH_5,22.894333,686.83


In [5]:
# Step 9.2B-1: Recreate inventory planning metrics

inventory_analysis = (
    df
    .groupby(["sku_id", "warehouse_id"])
    .agg(
        average_inventory=("inventory_level", "mean"),
        total_units_sold=("units_sold", "sum"),
        average_daily_demand=("units_sold", "mean"),
        demand_std=("units_sold", "std"),
        average_lead_time=("supplier_lead_time_days", "mean"),
        unit_cost=("unit_cost", "mean"),
        existing_reorder_point=("reorder_point", "mean")
    )
    .reset_index()
)

inventory_analysis["safety_stock"] = (
    1.645
    * inventory_analysis["demand_std"]
    * np.sqrt(inventory_analysis["average_lead_time"])
)

inventory_analysis["forecast_lead_time_demand"] = (
    forecast_summary["forecast_daily_demand"].values
    * inventory_analysis["average_lead_time"]
)

inventory_analysis["optimized_reorder_point"] = (
    inventory_analysis["forecast_lead_time_demand"]
    + inventory_analysis["safety_stock"]
)

inventory_analysis["consumption_value"] = (
    inventory_analysis["total_units_sold"]
    * inventory_analysis["unit_cost"]
)

inventory_analysis = inventory_analysis.sort_values(
    "consumption_value",
    ascending=False
).reset_index(drop=True)

inventory_analysis["cumulative_value_pct"] = (
    inventory_analysis["consumption_value"].cumsum()
    / inventory_analysis["consumption_value"].sum()
    * 100
)

inventory_analysis["abc_class"] = np.select(
    [
        inventory_analysis["cumulative_value_pct"] <= 80,
        inventory_analysis["cumulative_value_pct"] <= 95
    ],
    ["A", "B"],
    default="C"
)

inventory_analysis["demand_cv"] = (
    inventory_analysis["demand_std"]
    / inventory_analysis["average_daily_demand"]
)

inventory_analysis["xyz_class"] = np.select(
    [
        inventory_analysis["demand_cv"] <= 0.50,
        inventory_analysis["demand_cv"] <= 1.00
    ],
    ["X", "Y"],
    default="Z"
)

print("Inventory planning metrics recreated.")
print("SKU-Warehouse combinations:", len(inventory_analysis))

display(
    inventory_analysis[
        [
            "sku_id",
            "warehouse_id",
            "safety_stock",
            "optimized_reorder_point",
            "abc_class",
            "xyz_class"
        ]
    ].head(10)
)

Inventory planning metrics recreated.
SKU-Warehouse combinations: 250


,sku_id,warehouse_id,safety_stock,optimized_reorder_point,abc_class,xyz_class
0,SKU_38,WH_3,45.633851,217.855184,A,X
1,SKU_13,WH_1,44.194753,269.194753,A,X
2,SKU_23,WH_5,24.548924,83.849924,A,X
3,SKU_31,WH_4,24.079162,84.120162,A,X
4,SKU_1,WH_5,29.652850,122.851517,A,X
5,SKU_2,WH_1,54.920169,371.730169,A,X
6,SKU_24,WH_2,32.229326,135.079326,A,X
7,SKU_4,WH_2,53.123583,350.307916,A,X
8,SKU_11,WH_1,29.956108,93.956108,A,X
9,SKU_32,WH_5,42.960686,259.785686,A,X


In [6]:
# Step 9.2B: Combine inventory, forecast and reorder policy

replenishment_plan = (
    latest_inventory
    .merge(
        forecast_summary,
        on=["sku_id", "warehouse_id"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        inventory_analysis[
            [
                "sku_id",
                "warehouse_id",
                "average_lead_time",
                "safety_stock",
                "optimized_reorder_point",
                "existing_reorder_point",
                "abc_class",
                "xyz_class"
            ]
        ],
        on=["sku_id", "warehouse_id"],
        how="left",
        validate="one_to_one"
    )
)

print("Replenishment Planning Input Summary")
print("SKU-Warehouse combinations:", len(replenishment_plan))
print(
    "Missing forecast daily demand:",
    replenishment_plan["forecast_daily_demand"].isna().sum()
)
print(
    "Missing optimized ROP:",
    replenishment_plan["optimized_reorder_point"].isna().sum()
)

display(replenishment_plan.head(10))

Replenishment Planning Input Summary
SKU-Warehouse combinations: 250
Missing forecast daily demand: 0
Missing optimized ROP: 0


,sku_id,warehouse_id,inventory_as_of_date,current_inventory,forecast_daily_demand,total_forecast_demand,average_lead_time,safety_stock,optimized_reorder_point,existing_reorder_point,abc_class,xyz_class
0,SKU_25,WH_5,2024-12-30,373,20.150667,604.52,7.0,37.788577,178.843243,311.0,B,X
1,SKU_50,WH_3,2024-12-30,369,20.634667,619.04,11.0,49.112270,276.093603,217.0,A,X
2,SKU_18,WH_5,2024-12-30,766,23.759667,712.79,13.0,54.468296,363.343963,394.0,A,X
3,SKU_18,WH_4,2024-12-30,339,17.000000,510.00,6.0,36.928014,138.928014,250.0,C,X
4,SKU_18,WH_3,2024-12-30,442,17.182667,515.48,9.0,46.100659,200.744659,213.0,A,X
5,SKU_18,WH_2,2024-12-30,272,18.641333,559.24,10.0,49.365338,235.778672,270.0,A,X
6,SKU_18,WH_1,2024-12-30,407,28.000000,840.00,2.0,21.826387,77.826387,346.0,B,X
7,SKU_17,WH_5,2024-12-30,334,27.011667,810.35,8.0,41.743301,257.836634,321.0,B,X
8,SKU_17,WH_4,2024-12-30,362,25.404667,762.14,5.0,34.664175,161.687509,341.0,B,X
9,SKU_17,WH_3,2024-12-30,574,21.897000,656.91,3.0,25.262204,90.953204,339.0,A,X


In [7]:
# Step 9.3: Calculate lead-time demand and inventory coverage

replenishment_plan["forecast_lead_time_demand"] = (
    replenishment_plan["forecast_daily_demand"]
    * replenishment_plan["average_lead_time"]
)

replenishment_plan["current_inventory_coverage_days"] = (
    replenishment_plan["current_inventory"]
    / replenishment_plan["forecast_daily_demand"]
)

print("Replenishment Requirement Metrics\n")

print(
    "Average Forecast Lead-Time Demand:",
    round(replenishment_plan["forecast_lead_time_demand"].mean(), 2)
)

print(
    "Average Current Inventory Coverage:",
    round(replenishment_plan["current_inventory_coverage_days"].mean(), 2),
    "days"
)

display(
    replenishment_plan[
        [
            "sku_id",
            "warehouse_id",
            "current_inventory",
            "forecast_daily_demand",
            "average_lead_time",
            "forecast_lead_time_demand",
            "current_inventory_coverage_days",
            "safety_stock",
            "optimized_reorder_point"
        ]
    ].head(10)
)

Replenishment Requirement Metrics

Average Forecast Lead-Time Demand: 170.66
Average Current Inventory Coverage: 21.78 days


,sku_id,warehouse_id,current_inventory,forecast_daily_demand,average_lead_time,forecast_lead_time_demand,current_inventory_coverage_days,safety_stock,optimized_reorder_point
0,SKU_25,WH_5,373,20.150667,7.0,141.054667,18.510554,37.788577,178.843243
1,SKU_50,WH_3,369,20.634667,11.0,226.981333,17.882528,49.112270,276.093603
2,SKU_18,WH_5,766,23.759667,13.0,308.875667,32.239510,54.468296,363.343963
3,SKU_18,WH_4,339,17.000000,6.0,102.000000,19.941176,36.928014,138.928014
4,SKU_18,WH_3,442,17.182667,9.0,154.644000,25.723597,46.100659,200.744659
5,SKU_18,WH_2,272,18.641333,10.0,186.413333,14.591231,49.365338,235.778672
6,SKU_18,WH_1,407,28.000000,2.0,56.000000,14.535714,21.826387,77.826387
7,SKU_17,WH_5,334,27.011667,8.0,216.093333,12.365027,41.743301,257.836634
8,SKU_17,WH_4,362,25.404667,5.0,127.023333,14.249351,34.664175,161.687509
9,SKU_17,WH_3,574,21.897000,3.0,65.691000,26.213637,25.262204,90.953204


In [8]:
# Step 9.4: Generate replenishment recommendations

replenishment_plan["recommended_order_quantity"] = (
    replenishment_plan["optimized_reorder_point"]
    - replenishment_plan["current_inventory"]
).clip(lower=0)

replenishment_plan["replenishment_recommendation"] = np.select(
    [
        (
            replenishment_plan["current_inventory"]
            < replenishment_plan["forecast_lead_time_demand"]
        ),
        (
            replenishment_plan["current_inventory"]
            <= replenishment_plan["optimized_reorder_point"]
        )
    ],
    [
        "Urgent",
        "Reorder"
    ],
    default="Hold"
)

replenishment_plan["recommended_order_quantity"] = np.where(
    replenishment_plan["replenishment_recommendation"] == "Hold",
    0,
    replenishment_plan["recommended_order_quantity"]
)

print("Replenishment Recommendation Summary\n")

display(
    replenishment_plan["replenishment_recommendation"]
    .value_counts()
    .rename_axis("Recommendation")
    .reset_index(name="SKU_Warehouse_Count")
)

display(
    replenishment_plan[
        [
            "sku_id",
            "warehouse_id",
            "current_inventory",
            "forecast_daily_demand",
            "average_lead_time",
            "forecast_lead_time_demand",
            "safety_stock",
            "optimized_reorder_point",
            "recommended_order_quantity",
            "replenishment_recommendation"
        ]
    ]
    .sort_values(
        ["replenishment_recommendation", "recommended_order_quantity"],
        ascending=[True, False]
    )
    .head(15)
)

Replenishment Recommendation Summary



,Recommendation,SKU_Warehouse_Count
0,Hold,241
1,Reorder,6
2,Urgent,3


,sku_id,warehouse_id,current_inventory,forecast_daily_demand,average_lead_time,forecast_lead_time_demand,safety_stock,optimized_reorder_point,recommended_order_quantity,replenishment_recommendation
0,SKU_25,WH_5,373,20.150667,7.0,141.054667,37.788577,178.843243,0.0,Hold
1,SKU_50,WH_3,369,20.634667,11.0,226.981333,49.112270,276.093603,0.0,Hold
2,SKU_18,WH_5,766,23.759667,13.0,308.875667,54.468296,363.343963,0.0,Hold
3,SKU_18,WH_4,339,17.000000,6.0,102.000000,36.928014,138.928014,0.0,Hold
4,SKU_18,WH_3,442,17.182667,9.0,154.644000,46.100659,200.744659,0.0,Hold
5,SKU_18,WH_2,272,18.641333,10.0,186.413333,49.365338,235.778672,0.0,Hold
6,SKU_18,WH_1,407,28.000000,2.0,56.000000,21.826387,77.826387,0.0,Hold
7,SKU_17,WH_5,334,27.011667,8.0,216.093333,41.743301,257.836634,0.0,Hold
8,SKU_17,WH_4,362,25.404667,5.0,127.023333,34.664175,161.687509,0.0,Hold
9,SKU_17,WH_3,574,21.897000,3.0,65.691000,25.262204,90.953204,0.0,Hold


In [9]:
# PHASE 9 — Supply / Replenishment Planning
# Step 9.5: Review actionable replenishment recommendations

replenishment_plan["recommended_order_quantity"] = np.where(
    replenishment_plan["replenishment_recommendation"] == "Hold",
    0,
    np.ceil(replenishment_plan["recommended_order_quantity"])
).astype(int)

action_required = (
    replenishment_plan[
        replenishment_plan["replenishment_recommendation"].isin(
            ["Urgent", "Reorder"]
        )
    ]
    .copy()
)

action_required["priority"] = action_required[
    "replenishment_recommendation"
].map({
    "Urgent": 1,
    "Reorder": 2
})

action_required = action_required.sort_values(
    ["priority", "recommended_order_quantity"],
    ascending=[True, False]
)

print("Action Required — Urgent and Reorder Items")
print("Total combinations requiring action:", len(action_required))

display(
    action_required[
        [
            "sku_id",
            "warehouse_id",
            "abc_class",
            "xyz_class",
            "current_inventory",
            "forecast_lead_time_demand",
            "safety_stock",
            "optimized_reorder_point",
            "recommended_order_quantity",
            "replenishment_recommendation"
        ]
    ]
)

Action Required — Urgent and Reorder Items
Total combinations requiring action: 9


,sku_id,warehouse_id,abc_class,xyz_class,current_inventory,forecast_lead_time_demand,safety_stock,optimized_reorder_point,recommended_order_quantity,replenishment_recommendation
124,SKU_50,WH_4,A,X,236,321.850667,53.788065,375.638731,140,Urgent
85,SKU_2,WH_1,A,X,271,316.810000,54.920169,371.730169,101,Urgent
156,SKU_44,WH_3,A,X,200,208.000000,54.346432,262.346432,63,Urgent
80,SKU_3,WH_1,B,X,367,326.662000,56.830855,383.492855,17,Reorder
185,SKU_44,WH_4,C,X,302,263.388000,51.039296,314.427296,13,Reorder
229,SKU_36,WH_1,A,X,386,348.000000,50.363917,398.363917,13,Reorder
165,SKU_48,WH_5,B,X,369,323.924333,50.086054,374.010387,6,Reorder
208,SKU_27,WH_4,A,X,315,262.957333,57.256386,320.213719,6,Reorder
195,SKU_30,WH_2,A,X,335,283.816000,51.739400,335.555400,1,Reorder


In [10]:
# PHASE 9 — Supply / Replenishment Planning
# Step 9.6: Create the final replenishment recommendation table

final_replenishment_table = replenishment_plan[
    [
        "sku_id",
        "warehouse_id",
        "inventory_as_of_date",
        "abc_class",
        "xyz_class",
        "current_inventory",
        "forecast_daily_demand",
        "total_forecast_demand",
        "average_lead_time",
        "forecast_lead_time_demand",
        "safety_stock",
        "optimized_reorder_point",
        "existing_reorder_point",
        "recommended_order_quantity",
        "replenishment_recommendation"
    ]
].copy()

# Put Urgent items first, followed by Reorder and Hold
priority_order = {
    "Urgent": 1,
    "Reorder": 2,
    "Hold": 3
}

final_replenishment_table["priority"] = (
    final_replenishment_table["replenishment_recommendation"]
    .map(priority_order)
)

final_replenishment_table = (
    final_replenishment_table
    .sort_values(
        ["priority", "recommended_order_quantity"],
        ascending=[True, False]
    )
    .drop(columns="priority")
    .reset_index(drop=True)
)

print("Final Replenishment Recommendation Table")
print("Total combinations:", len(final_replenishment_table))

display(final_replenishment_table.head(15))

Final Replenishment Recommendation Table
Total combinations: 250


,sku_id,warehouse_id,inventory_as_of_date,abc_class,xyz_class,current_inventory,forecast_daily_demand,total_forecast_demand,average_lead_time,forecast_lead_time_demand,safety_stock,optimized_reorder_point,existing_reorder_point,recommended_order_quantity,replenishment_recommendation
0,SKU_50,WH_4,2024-12-30,A,X,236,22.989333,689.68,14.0,321.850667,53.788065,375.638731,218.0,140,Urgent
1,SKU_2,WH_1,2024-12-30,A,X,271,24.370000,731.10,13.0,316.810000,54.920169,371.730169,228.0,101,Urgent
2,SKU_44,WH_3,2024-12-30,A,X,200,16.000000,480.00,13.0,208.000000,54.346432,262.346432,201.0,63,Urgent
3,SKU_3,WH_1,2024-12-30,B,X,367,23.333000,699.99,14.0,326.662000,56.830855,383.492855,240.0,17,Reorder
4,SKU_44,WH_4,2024-12-30,C,X,302,21.949000,658.47,12.0,263.388000,51.039296,314.427296,242.0,13,Reorder
5,SKU_36,WH_1,2024-12-30,A,X,386,29.000000,870.00,12.0,348.000000,50.363917,398.363917,270.0,13,Reorder
6,SKU_48,WH_5,2024-12-30,B,X,369,29.447667,883.43,11.0,323.924333,50.086054,374.010387,275.0,6,Reorder
7,SKU_27,WH_4,2024-12-30,A,X,315,18.782667,563.48,14.0,262.957333,57.256386,320.213719,289.0,6,Reorder
8,SKU_30,WH_2,2024-12-30,A,X,335,23.651333,709.54,12.0,283.816000,51.739400,335.555400,231.0,1,Reorder
9,SKU_25,WH_5,2024-12-30,B,X,373,20.150667,604.52,7.0,141.054667,37.788577,178.843243,311.0,0,Hold


In [11]:
# PHASE 9 — Supply / Replenishment Planning
# Step 9.7: Export final recommendations to Excel

output_file = "Final_Replenishment_Recommendations.xlsx"

final_replenishment_table.to_excel(
    output_file,
    sheet_name="Replenishment Plan",
    index=False
)

print("Export completed:", output_file)
print("Rows exported:", len(final_replenishment_table))

Export completed: Final_Replenishment_Recommendations.xlsx
Rows exported: 250


In [12]:
# PHASE 10 — Supplier Analytics
# Step 10.1: Inspect supplier-related columns

supplier_columns = [
    column
    for column in df.columns
    if "supplier" in column.lower()
]

print("Supplier-related columns:")
print(supplier_columns)

display(df[supplier_columns].head(10))

Supplier-related columns:
['supplier_id', 'supplier_lead_time_days']


,supplier_id,supplier_lead_time_days
0,SUP_8,14
1,SUP_8,14
2,SUP_8,14
3,SUP_8,14
4,SUP_8,14
5,SUP_8,14
6,SUP_8,14
7,SUP_8,14
8,SUP_8,14
9,SUP_8,14


In [13]:
# PHASE 10 — Supplier Analytics
# Step 10.2: Build the supplier scorecard

supplier_scorecard = (
    df
    .groupby("supplier_id")
    .agg(
        average_lead_time=("supplier_lead_time_days", "mean"),
        minimum_lead_time=("supplier_lead_time_days", "min"),
        maximum_lead_time=("supplier_lead_time_days", "max"),
        sku_count=("sku_id", "nunique"),
        warehouse_count=("warehouse_id", "nunique"),
        total_units_sold=("units_sold", "sum"),
        average_inventory=("inventory_level", "mean"),
        average_order_quantity=("order_quantity", "mean")
    )
    .reset_index()
)

supplier_scorecard = supplier_scorecard.sort_values(
    "average_lead_time",
    ascending=False
).reset_index(drop=True)

print("Supplier Scorecard")
print("Number of suppliers:", len(supplier_scorecard))

display(supplier_scorecard)

Supplier Scorecard
Number of suppliers: 10


,supplier_id,average_lead_time,minimum_lead_time,maximum_lead_time,sku_count,warehouse_count,total_units_sold,average_inventory,average_order_quantity
0,SUP_4,8.640000,3,14,22,5,183147,472.917699,19.349918
1,SUP_8,8.500000,2,14,22,5,205407,468.984638,19.180333
2,SUP_2,8.423077,2,14,21,5,190999,464.430242,19.456797
3,SUP_3,8.260870,2,14,20,5,168017,456.394640,19.263967
4,SUP_9,8.187500,2,14,16,5,117259,464.152911,19.188699
5,SUP_1,8.107143,3,14,23,5,204249,464.689922,19.088063
6,SUP_6,7.900000,2,14,16,5,146684,474.610685,19.364521
7,SUP_7,7.558824,2,14,29,5,248830,494.258743,19.281467
8,SUP_10,7.407407,2,14,21,5,197242,472.659158,19.231355
9,SUP_5,6.956522,2,13,21,5,168145,472.053484,19.338535


In [14]:
# PHASE 10 — Supplier Analytics
# Step 10.3: Identify long-lead suppliers

lead_time_threshold = 8

supplier_scorecard["lead_time_category"] = np.where(
    supplier_scorecard["average_lead_time"] > lead_time_threshold,
    "Long Lead Time",
    "Standard Lead Time"
)

supplier_scorecard = supplier_scorecard.sort_values(
    "average_lead_time",
    ascending=False
).reset_index(drop=True)

print("Supplier Lead-Time Classification\n")

display(
    supplier_scorecard[
        [
            "supplier_id",
            "average_lead_time",
            "minimum_lead_time",
            "maximum_lead_time",
            "sku_count",
            "lead_time_category"
        ]
    ]
)

print("\nLead-Time Category Summary")

display(
    supplier_scorecard["lead_time_category"]
    .value_counts()
    .rename_axis("Lead_Time_Category")
    .reset_index(name="Supplier_Count")
)

Supplier Lead-Time Classification



,supplier_id,average_lead_time,minimum_lead_time,maximum_lead_time,sku_count,lead_time_category
0,SUP_4,8.640000,3,14,22,Long Lead Time
1,SUP_8,8.500000,2,14,22,Long Lead Time
2,SUP_2,8.423077,2,14,21,Long Lead Time
3,SUP_3,8.260870,2,14,20,Long Lead Time
4,SUP_9,8.187500,2,14,16,Long Lead Time
5,SUP_1,8.107143,3,14,23,Long Lead Time
6,SUP_6,7.900000,2,14,16,Standard Lead Time
7,SUP_7,7.558824,2,14,29,Standard Lead Time
8,SUP_10,7.407407,2,14,21,Standard Lead Time
9,SUP_5,6.956522,2,13,21,Standard Lead Time



Lead-Time Category Summary


,Lead_Time_Category,Supplier_Count
0,Long Lead Time,6
1,Standard Lead Time,4


In [15]:
# PHASE 10 — Supplier Analytics
# Step 10.4: Analyze supplier concentration

supplier_scorecard["sales_volume_share_pct"] = (
    supplier_scorecard["total_units_sold"]
    / supplier_scorecard["total_units_sold"].sum()
    * 100
)

supplier_scorecard["sku_portfolio_share_pct"] = (
    supplier_scorecard["sku_count"]
    / supplier_scorecard["sku_count"].sum()
    * 100
)

supplier_scorecard = supplier_scorecard.sort_values(
    "total_units_sold",
    ascending=False
).reset_index(drop=True)

supplier_scorecard["cumulative_sales_share_pct"] = (
    supplier_scorecard["sales_volume_share_pct"].cumsum()
)

print("Supplier Concentration Analysis\n")

display(
    supplier_scorecard[
        [
            "supplier_id",
            "sku_count",
            "total_units_sold",
            "sales_volume_share_pct",
            "cumulative_sales_share_pct",
            "average_lead_time",
            "lead_time_category"
        ]
    ]
)

Supplier Concentration Analysis



,supplier_id,sku_count,total_units_sold,sales_volume_share_pct,cumulative_sales_share_pct,average_lead_time,lead_time_category
0,SUP_7,29,248830,13.597424,13.597424,7.558824,Standard Lead Time
1,SUP_8,22,205407,11.224555,24.821979,8.500000,Long Lead Time
2,SUP_1,23,204249,11.161276,35.983254,8.107143,Long Lead Time
3,SUP_10,21,197242,10.778375,46.761630,7.407407,Standard Lead Time
4,SUP_2,21,190999,10.437224,57.198853,8.423077,Long Lead Time
5,SUP_4,22,183147,10.008148,67.207001,8.640000,Long Lead Time
6,SUP_5,21,168145,9.188357,76.395358,6.956522,Standard Lead Time
7,SUP_3,20,168017,9.181362,85.576720,8.260870,Long Lead Time
8,SUP_6,16,146684,8.015611,93.592331,7.900000,Standard Lead Time
9,SUP_9,16,117259,6.407669,100.000000,8.187500,Long Lead Time


In [16]:
# PHASE 10 — Supplier Analytics
# Step 10.5: Flag suppliers for planning review

supplier_scorecard["volume_category"] = np.where(
    supplier_scorecard["sales_volume_share_pct"] >= 10,
    "High Volume",
    "Standard Volume"
)

supplier_scorecard["supplier_review_flag"] = np.select(
    [
        (
            (supplier_scorecard["volume_category"] == "High Volume")
            &
            (supplier_scorecard["lead_time_category"] == "Long Lead Time")
        ),
        (
            supplier_scorecard["volume_category"] == "High Volume"
        ),
        (
            supplier_scorecard["lead_time_category"] == "Long Lead Time"
        )
    ],
    [
        "High Volume + Long Lead Time",
        "High Volume",
        "Long Lead Time"
    ],
    default="Standard Review"
)

print("Supplier Planning Review Summary\n")

display(
    supplier_scorecard[
        [
            "supplier_id",
            "sales_volume_share_pct",
            "average_lead_time",
            "sku_count",
            "volume_category",
            "lead_time_category",
            "supplier_review_flag"
        ]
    ]
    .sort_values(
        ["sales_volume_share_pct", "average_lead_time"],
        ascending=[False, False]
    )
)

Supplier Planning Review Summary



,supplier_id,sales_volume_share_pct,average_lead_time,sku_count,volume_category,lead_time_category,supplier_review_flag
0,SUP_7,13.597424,7.558824,29,High Volume,Standard Lead Time,High Volume
1,SUP_8,11.224555,8.500000,22,High Volume,Long Lead Time,High Volume + Long Lead Time
2,SUP_1,11.161276,8.107143,23,High Volume,Long Lead Time,High Volume + Long Lead Time
3,SUP_10,10.778375,7.407407,21,High Volume,Standard Lead Time,High Volume
4,SUP_2,10.437224,8.423077,21,High Volume,Long Lead Time,High Volume + Long Lead Time
5,SUP_4,10.008148,8.640000,22,High Volume,Long Lead Time,High Volume + Long Lead Time
6,SUP_5,9.188357,6.956522,21,Standard Volume,Standard Lead Time,Standard Review
7,SUP_3,9.181362,8.260870,20,Standard Volume,Long Lead Time,Long Lead Time
8,SUP_6,8.015611,7.900000,16,Standard Volume,Standard Lead Time,Standard Review
9,SUP_9,6.407669,8.187500,16,Standard Volume,Long Lead Time,Long Lead Time


In [17]:
# PHASE 10 — Supplier Analytics
# Step 10.6: Summarize supplier planning categories

supplier_review_summary = (
    supplier_scorecard
    .groupby("supplier_review_flag")
    .agg(
        supplier_count=("supplier_id", "nunique"),
        average_lead_time=("average_lead_time", "mean"),
        average_sales_volume_share=("sales_volume_share_pct", "mean"),
        total_skus_associated=("sku_count", "sum")
    )
    .reset_index()
    .sort_values("supplier_count", ascending=False)
)

print("PHASE 10 — SUPPLIER ANALYTICS SUMMARY")

display(supplier_review_summary)

PHASE 10 — SUPPLIER ANALYTICS SUMMARY


,supplier_review_flag,supplier_count,average_lead_time,average_sales_volume_share,total_skus_associated
1,High Volume + Long Lead Time,4,8.417555,10.707800,88
0,High Volume,2,7.483115,12.187899,50
2,Long Lead Time,2,8.224185,7.794516,36
3,Standard Review,2,7.428261,8.601984,37


In [18]:
# PHASE 11 — Logistics / Warehouse Operations
# Step 11.1: Build the warehouse performance summary

warehouse_summary = (
    df
    .groupby("warehouse_id")
    .agg(
        sku_count=("sku_id", "nunique"),
        average_inventory=("inventory_level", "mean"),
        latest_inventory=("inventory_level", "last"),
        total_units_sold=("units_sold", "sum"),
        average_daily_demand=("units_sold", "mean"),
        average_lead_time=("supplier_lead_time_days", "mean"),
        stockout_days=("stockout_flag", "sum")
    )
    .reset_index()
)

print("PHASE 11 — WAREHOUSE PERFORMANCE SUMMARY")
print("Warehouses:", len(warehouse_summary))

display(
    warehouse_summary.sort_values(
        "total_units_sold",
        ascending=False
    )
)

PHASE 11 — WAREHOUSE PERFORMANCE SUMMARY


Warehouses: 5


,warehouse_id,sku_count,average_inventory,latest_inventory,total_units_sold,average_daily_demand,average_lead_time,stockout_days
0,WH_1,50,466.056219,609,366902,20.104219,8.00,0
2,WH_3,50,455.559068,369,366110,20.060822,8.22,0
1,WH_2,50,491.489644,275,366078,20.059068,8.46,0
3,WH_4,50,477.390301,236,365775,20.042466,8.48,0
4,WH_5,50,467.116329,275,365114,20.006247,6.76,0


In [22]:
# PHASE 11 — Logistics / Warehouse Operations
# Step 11.2 : Calculate total latest inventory per warehouse

latest_date = df["date"].max()

latest_warehouse_inventory = (
    df[df["date"] == latest_date]
    .groupby("warehouse_id")
    .agg(
        sku_count=("sku_id", "nunique"),
        total_latest_inventory=("inventory_level", "sum")
    )
    .reset_index()
)

print("Latest Warehouse Inventory Summary")
print("Inventory as of:", latest_date)
print("Warehouse count:", len(latest_warehouse_inventory))

display(latest_warehouse_inventory)

Latest Warehouse Inventory Summary
Inventory as of: 2024-12-30 00:00:00
Warehouse count: 5


,warehouse_id,sku_count,total_latest_inventory
0,WH_1,50,22110
1,WH_2,50,24504
2,WH_3,50,22152
3,WH_4,50,22242
4,WH_5,50,23011


In [23]:
# PHASE 11 — Logistics / Warehouse Operations
# Step 11.3: Calculate warehouse inventory coverage

latest_date = df["date"].max()

warehouse_demand = (
    df[df["date"] == latest_date]
    .groupby("warehouse_id")
    .agg(
        daily_demand=("units_sold", "sum"),
        sku_count=("sku_id", "nunique")
    )
    .reset_index()
)

warehouse_inventory_coverage = (
    latest_warehouse_inventory
    .merge(
        warehouse_demand,
        on="warehouse_id",
        how="left",
        validate="one_to_one"
    )
)

warehouse_inventory_coverage["inventory_coverage_days"] = (
    warehouse_inventory_coverage["total_latest_inventory"]
    / warehouse_inventory_coverage["daily_demand"]
)

print("Warehouse Inventory Coverage Summary")

display(
    warehouse_inventory_coverage.sort_values(
        "inventory_coverage_days",
        ascending=True
    )
)

Warehouse Inventory Coverage Summary


,warehouse_id,sku_count_x,total_latest_inventory,daily_demand,sku_count_y,inventory_coverage_days
0,WH_1,50,22110,1041,50,21.239193
2,WH_3,50,22152,1039,50,21.320500
3,WH_4,50,22242,970,50,22.929897
4,WH_5,50,23011,998,50,23.057114
1,WH_2,50,24504,975,50,25.132308


In [27]:
# PHASE 11 — Logistics / Warehouse Operations
# Step 11.4: Refine warehouse inventory coverage

warehouse_daily_demand = (
    df
    .groupby(["date", "warehouse_id"])["units_sold"]
    .sum()
    .reset_index(name="daily_demand")
)

warehouse_average_demand = (
    warehouse_daily_demand
    .groupby("warehouse_id")
    .agg(
        average_daily_demand=("daily_demand", "mean")
    )
    .reset_index()
)

warehouse_inventory_coverage = (
    latest_warehouse_inventory[
        ["warehouse_id", "total_latest_inventory"]
    ]
    .merge(
        warehouse_average_demand,
        on="warehouse_id",
        how="left",
        validate="one_to_one"
    )
)

warehouse_inventory_coverage["inventory_coverage_days"] = (
    warehouse_inventory_coverage["total_latest_inventory"]
    / warehouse_inventory_coverage["average_daily_demand"]
)

display(
    warehouse_inventory_coverage.sort_values(
        "inventory_coverage_days"
    )
)

,warehouse_id,total_latest_inventory,average_daily_demand,inventory_coverage_days
0,WH_1,22110,1005.210959,21.995383
2,WH_3,22152,1003.041096,22.084838
3,WH_4,22242,1002.123288,22.194874
4,WH_5,23011,1000.312329,23.003815
1,WH_2,24504,1002.953425,24.431842


In [28]:
# PHASE 11 — Logistics / Warehouse Operations
# Step 11.5: Classify warehouse replenishment priority

warehouse_inventory_coverage["replenishment_priority"] = np.select(
    [
        warehouse_inventory_coverage["inventory_coverage_days"] < 20,
        warehouse_inventory_coverage["inventory_coverage_days"] < 25
    ],
    [
        "Urgent Review",
        "Monitor"
    ],
    default="Healthy Coverage"
)

print("Warehouse Replenishment Priority")

display(
    warehouse_inventory_coverage[
        [
            "warehouse_id",
            "total_latest_inventory",
            "average_daily_demand",
            "inventory_coverage_days",
            "replenishment_priority"
        ]
    ].sort_values("inventory_coverage_days")
)

print("\nPriority Summary")

display(
    warehouse_inventory_coverage["replenishment_priority"]
    .value_counts()
    .rename_axis("Replenishment_Priority")
    .reset_index(name="Warehouse_Count")
)

Warehouse Replenishment Priority


,warehouse_id,total_latest_inventory,average_daily_demand,inventory_coverage_days,replenishment_priority
0,WH_1,22110,1005.210959,21.995383,Monitor
2,WH_3,22152,1003.041096,22.084838,Monitor
3,WH_4,22242,1002.123288,22.194874,Monitor
4,WH_5,23011,1000.312329,23.003815,Monitor
1,WH_2,24504,1002.953425,24.431842,Monitor



Priority Summary


,Replenishment_Priority,Warehouse_Count
0,Monitor,5


In [33]:
# PHASE 11 — Logistics / Warehouse Operations
# Step 11.6A: Recreate stock-risk metrics for warehouse analysis

# 1. Calculate days of inventory
inventory_analysis["days_of_inventory"] = (
    inventory_analysis["average_inventory"]
    / inventory_analysis["average_daily_demand"].replace(0, np.nan)
)

# 2. Recreate the stockout indicator from the source dataset
stockout_summary = (
    df.groupby(["sku_id", "warehouse_id"])
    .agg(
        stockout_days=("stockout_flag", "sum")
    )
    .reset_index()
)

# 3. Attach stockout observations to inventory analysis
inventory_analysis = inventory_analysis.drop(
    columns=["stockout_days"],
    errors="ignore"
)

inventory_analysis = inventory_analysis.merge(
    stockout_summary,
    on=["sku_id", "warehouse_id"],
    how="left",
    validate="one_to_one"
)

# 4. Identify combinations with stockout observations
# or average inventory below the existing reorder point
inventory_analysis["at_risk_flag"] = (
    (inventory_analysis["stockout_days"] > 0)
    |
    (
        inventory_analysis["average_inventory"]
        < inventory_analysis["existing_reorder_point"]
    )
)

print("Stock-risk metrics recreated successfully.")
print("SKU-Warehouse combinations:", len(inventory_analysis))

display(
    inventory_analysis[
        [
            "sku_id",
            "warehouse_id",
            "days_of_inventory",
            "stockout_days",
            "at_risk_flag"
        ]
    ].head(10)
)

Stock-risk metrics recreated successfully.
SKU-Warehouse combinations: 250


,sku_id,warehouse_id,days_of_inventory,stockout_days,at_risk_flag
0,SKU_38,WH_3,24.257557,0,False
1,SKU_13,WH_1,21.546088,0,False
2,SKU_23,WH_5,26.586674,0,False
3,SKU_31,WH_4,21.346781,0,False
4,SKU_1,WH_5,26.813029,0,False
5,SKU_2,WH_1,20.797264,0,False
6,SKU_24,WH_2,27.428904,0,False
7,SKU_4,WH_2,23.924926,0,False
8,SKU_11,WH_1,24.260159,0,False
9,SKU_32,WH_5,26.915857,0,False


In [34]:
# PHASE 11 — Logistics / Warehouse Operations
# Step 11.6B: Summarize SKU-level stock risk by warehouse

warehouse_stock_risk = (
    inventory_analysis
    .groupby("warehouse_id")
    .agg(
        total_sku_combinations=("sku_id", "nunique"),
        at_risk_combinations=("at_risk_flag", "sum"),
        stockout_observations=("stockout_days", "sum"),
        average_inventory_coverage=("days_of_inventory", "mean")
    )
    .reset_index()
)

warehouse_stock_risk["at_risk_pct"] = (
    warehouse_stock_risk["at_risk_combinations"]
    / warehouse_stock_risk["total_sku_combinations"]
    * 100
)

warehouse_stock_risk = warehouse_stock_risk.sort_values(
    "at_risk_pct",
    ascending=False
).reset_index(drop=True)

print("Warehouse Stock-Risk Summary")

display(warehouse_stock_risk)

Warehouse Stock-Risk Summary


,warehouse_id,total_sku_combinations,at_risk_combinations,stockout_observations,average_inventory_coverage,at_risk_pct
0,WH_1,50,0,0,23.191844,0.0
1,WH_2,50,0,0,24.509639,0.0
2,WH_3,50,0,0,22.712657,0.0
3,WH_4,50,0,0,23.828708,0.0
4,WH_5,50,0,0,23.346282,0.0


In [35]:
# PHASE 11 — Logistics / Warehouse Operations
# Step 11.7: Evaluate current warehouse inventory risk

warehouse_current_risk = warehouse_inventory_coverage.copy()

warehouse_current_risk["below_20_day_threshold"] = (
    warehouse_current_risk["inventory_coverage_days"] < 20
)

warehouse_current_risk["replenishment_priority"] = np.select(
    [
        warehouse_current_risk["inventory_coverage_days"] < 20,
        warehouse_current_risk["inventory_coverage_days"] < 25
    ],
    [
        "Urgent Review",
        "Monitor"
    ],
    default="Healthy Coverage"
)

print("Current Warehouse Replenishment Risk")

display(
    warehouse_current_risk.sort_values(
        "inventory_coverage_days"
    )
)

print("\nReplenishment Priority Summary")

display(
    warehouse_current_risk["replenishment_priority"]
    .value_counts()
    .rename_axis("Replenishment_Priority")
    .reset_index(name="Warehouse_Count")
)

Current Warehouse Replenishment Risk


,warehouse_id,total_latest_inventory,average_daily_demand,inventory_coverage_days,replenishment_priority,below_20_day_threshold
0,WH_1,22110,1005.210959,21.995383,Monitor,False
2,WH_3,22152,1003.041096,22.084838,Monitor,False
3,WH_4,22242,1002.123288,22.194874,Monitor,False
4,WH_5,23011,1000.312329,23.003815,Monitor,False
1,WH_2,24504,1002.953425,24.431842,Monitor,False



Replenishment Priority Summary


,Replenishment_Priority,Warehouse_Count
0,Monitor,5


In [36]:
# PHASE 11 — Logistics / Warehouse Operations
# Step 11.8: Compare warehouse inventory against demand

warehouse_comparison = warehouse_current_risk[
    [
        "warehouse_id",
        "total_latest_inventory",
        "average_daily_demand",
        "inventory_coverage_days"
    ]
].copy()

warehouse_comparison["inventory_share_pct"] = (
    warehouse_comparison["total_latest_inventory"]
    / warehouse_comparison["total_latest_inventory"].sum()
    * 100
)

warehouse_comparison["demand_share_pct"] = (
    warehouse_comparison["average_daily_demand"]
    / warehouse_comparison["average_daily_demand"].sum()
    * 100
)

warehouse_comparison["inventory_demand_share_gap"] = (
    warehouse_comparison["inventory_share_pct"]
    - warehouse_comparison["demand_share_pct"]
)

warehouse_comparison = warehouse_comparison.sort_values(
    "inventory_demand_share_gap",
    ascending=False
).reset_index(drop=True)

print("Warehouse Inventory vs Demand Comparison")

display(warehouse_comparison.round(2))

Warehouse Inventory vs Demand Comparison


,warehouse_id,total_latest_inventory,average_daily_demand,inventory_coverage_days,inventory_share_pct,demand_share_pct,inventory_demand_share_gap
0,WH_2,24504,1002.95,24.43,21.49,20.00,1.49
1,WH_5,23011,1000.31,23.00,20.18,19.95,0.23
2,WH_4,22242,1002.12,22.19,19.51,19.99,-0.48
3,WH_3,22152,1003.04,22.08,19.43,20.01,-0.58
4,WH_1,22110,1005.21,22.00,19.39,20.05,-0.66


In [37]:
# PHASE 11 — Logistics / Warehouse Operations
# Step 11.9: Classify warehouse inventory allocation

warehouse_comparison["allocation_review"] = np.select(
    [
        warehouse_comparison["inventory_demand_share_gap"] > 1.0,
        warehouse_comparison["inventory_demand_share_gap"] < -0.5
    ],
    [
        "Review Higher Inventory Share",
        "Review Lower Inventory Share"
    ],
    default="Balanced"
)

print("Warehouse Allocation Review")

display(
    warehouse_comparison[
        [
            "warehouse_id",
            "inventory_share_pct",
            "demand_share_pct",
            "inventory_demand_share_gap",
            "allocation_review"
        ]
    ].sort_values(
        "inventory_demand_share_gap",
        ascending=False
    ).round(2)
)

print("\nAllocation Review Summary")

display(
    warehouse_comparison["allocation_review"]
    .value_counts()
    .rename_axis("Allocation_Review")
    .reset_index(name="Warehouse_Count")
)

Warehouse Allocation Review


,warehouse_id,inventory_share_pct,demand_share_pct,inventory_demand_share_gap,allocation_review
0,WH_2,21.49,20.00,1.49,Review Higher Inventory Share
1,WH_5,20.18,19.95,0.23,Balanced
2,WH_4,19.51,19.99,-0.48,Balanced
3,WH_3,19.43,20.01,-0.58,Review Lower Inventory Share
4,WH_1,19.39,20.05,-0.66,Review Lower Inventory Share



Allocation Review Summary


,Allocation_Review,Warehouse_Count
0,Balanced,2
1,Review Lower Inventory Share,2
2,Review Higher Inventory Share,1


In [38]:
# PHASE 11 — Logistics / Warehouse Operations
# Step 11.10: Export warehouse analysis results to Excel

output_file = "Warehouse_Operations_Analysis.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    warehouse_inventory_coverage.to_excel(
        writer,
        sheet_name="Warehouse Coverage",
        index=False
    )

    warehouse_stock_risk.to_excel(
        writer,
        sheet_name="Stock Risk",
        index=False
    )

    warehouse_current_risk.to_excel(
        writer,
        sheet_name="Current Risk",
        index=False
    )

    warehouse_comparison.to_excel(
        writer,
        sheet_name="Inventory vs Demand",
        index=False
    )

print("Export completed:", output_file)
print("Sheets exported: 4")

Export completed: Warehouse_Operations_Analysis.xlsx
Sheets exported: 4


In [39]:
# PHASE 12 — Project Validation and Power BI Preparation
# Step 12.1: Validate final reporting datasets

validation_results = []

datasets_to_validate = {
    "Final Replenishment": final_replenishment_table,
    "Forecast Summary": forecast_summary,
    "Supplier Scorecard": supplier_scorecard,
    "Warehouse Comparison": warehouse_comparison
}

for dataset_name, dataset in datasets_to_validate.items():

    validation_results.append({
        "Dataset": dataset_name,
        "Rows": len(dataset),
        "Columns": len(dataset.columns),
        "Duplicate Rows": dataset.duplicated().sum(),
        "Missing Cells": dataset.isna().sum().sum()
    })

validation_summary = pd.DataFrame(validation_results)

print("PHASE 12.1 — DATASET VALIDATION SUMMARY")

display(validation_summary)

PHASE 12.1 — DATASET VALIDATION SUMMARY


,Dataset,Rows,Columns,Duplicate Rows,Missing Cells
0,Final Replenishment,250,15,0,0
1,Forecast Summary,250,4,0,0
2,Supplier Scorecard,10,15,0,0
3,Warehouse Comparison,5,8,0,0


In [40]:
# PHASE 12 — Project Validation and Power BI Preparation
# Step 12.2: Validate business-key uniqueness

key_validation = {
    "Final Replenishment": (
        final_replenishment_table,
        ["sku_id", "warehouse_id"]
    ),
    "Forecast Summary": (
        forecast_summary,
        ["sku_id", "warehouse_id"]
    ),
    "Supplier Scorecard": (
        supplier_scorecard,
        ["supplier_id"]
    ),
    "Warehouse Comparison": (
        warehouse_comparison,
        ["warehouse_id"]
    )
}

key_results = []

for dataset_name, (dataset, key_columns) in key_validation.items():

    duplicate_keys = dataset.duplicated(
        subset=key_columns
    ).sum()

    missing_keys = dataset[key_columns].isna().any(axis=1).sum()

    key_results.append({
        "Dataset": dataset_name,
        "Business Key": " + ".join(key_columns),
        "Duplicate Keys": duplicate_keys,
        "Rows With Missing Keys": missing_keys,
        "Key Validation": (
            "Passed"
            if duplicate_keys == 0 and missing_keys == 0
            else "Review Required"
        )
    })

key_validation_summary = pd.DataFrame(key_results)

print("PHASE 12.2 — BUSINESS KEY VALIDATION")

display(key_validation_summary)

PHASE 12.2 — BUSINESS KEY VALIDATION


,Dataset,Business Key,Duplicate Keys,Rows With Missing Keys,Key Validation
0,Final Replenishment,sku_id + warehouse_id,0,0,Passed
1,Forecast Summary,sku_id + warehouse_id,0,0,Passed
2,Supplier Scorecard,supplier_id,0,0,Passed
3,Warehouse Comparison,warehouse_id,0,0,Passed


In [41]:
# PHASE 12 — Project Validation and Power BI Preparation
# Step 12.3: Export validated reporting datasets

output_file = "PowerBI_Reporting_Datasets.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    final_replenishment_table.to_excel(
        writer,
        sheet_name="Replenishment",
        index=False
    )

    forecast_summary.to_excel(
        writer,
        sheet_name="Forecast Summary",
        index=False
    )

    supplier_scorecard.to_excel(
        writer,
        sheet_name="Supplier Scorecard",
        index=False
    )

    warehouse_comparison.to_excel(
        writer,
        sheet_name="Warehouse Comparison",
        index=False
    )

print("Power BI reporting export completed!")
print("Output file:", output_file)
print("Worksheets exported: 4")

print("\nRows exported:")
print("Replenishment:", len(final_replenishment_table))
print("Forecast Summary:", len(forecast_summary))
print("Supplier Scorecard:", len(supplier_scorecard))
print("Warehouse Comparison:", len(warehouse_comparison))

Power BI reporting export completed!
Output file: PowerBI_Reporting_Datasets.xlsx
Worksheets exported: 4

Rows exported:
Replenishment: 250
Forecast Summary: 250
Supplier Scorecard: 10
Warehouse Comparison: 5
